In [1]:
import json
import itertools
from pathlib import Path
import pandas as pd

In [2]:
def load_results_summaries(base_dir, direction_pairs, system_names):
    """
    Loads all result summaries from a directory structure.

    Args:
        base_dir (str or Path): The base directory for the evaluation outputs.
        direction_pairs (list): A list of language direction strings (e.g., 'en_de').
        system_names (list): A list of system name strings.

    Returns:
        dict: A nested dictionary containing the loaded data, structured as
              {direction: {system: [results]}}.
    """
    base_path = Path(base_dir)
    all_results = {}

    # Use itertools.product to cleanly iterate over all combinations
    for direction, system in itertools.product(direction_pairs, system_names):
        summary_path = base_path / system / direction / 'results_summary.jsonl'
        winoST_general = base_path / system / direction / 'eval.en.jsonl'
        winoST_anti = base_path / system / direction / 'eval.en_anti.jsonl'
        winoST_pro = base_path / system / direction / 'eval.en_pro.jsonl'
        
        # Initialize the nested dictionary structure
        if direction not in all_results:
            all_results[direction] = {}

        try:
            with summary_path.open('r', encoding='utf-8') as f:
                all_results[direction][system] = [json.loads(line) for line in f]

            with winoST_general.open('r', encoding='utf-8') as f:
                winoST_general = [json.loads(line) for line in f][0]
                winoST_general = {f"{k}_general": v for k, v in winoST_general.items() }
                all_results[direction][system][0].update(winoST_general)

            with winoST_anti.open('r', encoding='utf-8') as f:
                winoST_anti = [json.loads(line) for line in f][0]
                winoST_anti = {f"{k}_anti": v for k, v in winoST_anti.items() }
                all_results[direction][system][0].update(winoST_anti)
                
            with winoST_pro.open('r', encoding='utf-8') as f:
                winoST_pro = [json.loads(line) for line in f][0]
                winoST_pro = {f"{k}_pro": v for k, v in winoST_pro.items() }
                all_results[direction][system][0].update(winoST_pro)
                
        except FileNotFoundError:
            print(f"Warning: File not found, skipping: {summary_path}")
            all_results[direction][system] = None # Or [] if you prefer an empty list
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON in {summary_path}: {e}")
            all_results[direction][system] = None

    return all_results

In [3]:
def convert_results_to_dataframe(results_data):
    """
    Converts the nested dictionary of results into a single pandas DataFrame.

    Each row in the DataFrame corresponds to a single entry from a .jsonl file,
    augmented with 'direction' and 'system' columns to preserve its origin.

    Args:
        results_data (dict): The nested dictionary produced by the 
                             load_results_summaries function.

    Returns:
        pandas.DataFrame: A tidy DataFrame containing all results.
    """
    # Use a list comprehension for a fast and memory-efficient approach
    # This creates a flat list of records, where each record is a dictionary
    # that includes the original data plus the direction and system.
    all_records = [
        {
            'direction': direction,
            'system': system,
            **record  # Unpack the original record's key-value pairs
        }
        for direction, systems in results_data.items()
        for system, records in systems.items()
        if records is not None  # Gracefully skip any files that were not found
        for record in records
    ]

    if not all_records:
        print("No records were found to create a DataFrame.")
        return pd.DataFrame()

    # Convert the list of dictionaries directly into a DataFrame
    df = pd.DataFrame(all_records)

    # Reorder columns to have identifying info first, for better readability
    # Get all columns from the original data, excluding our added keys
    original_cols = [col for col in df.columns if col not in ['direction', 'system']]
    # Create the desired column order
    preferred_order = ['direction', 'system'] + original_cols
    df = df[preferred_order]

    return df

In [43]:
BASE_DIR = '/Users/javigg/Desktop/hearing2translate/evaluation/output_evals/winoST/'
DIRECTION_PAIRS = ['en_de', 'en_es', 'en_fr', 'en_it', 'en_pt']
SYSTEM_NAMES = ['seamlessm4t', 'canary-v2', 'owsm4.0-ctc',
                'aya_whisper', 'gemma_whisper', 'tower_whisper', 
                'aya_seamlessm4t', 'gemma_seamlessm4t', 'tower_seamlessm4t',
                'aya_canary-v2', 'gemma_canary-v2', 'tower_canary-v2',
                'aya_owsm4.0-ctc', 'gemma_owsm4.0-ctc', 'tower_owsm4.0-ctc',
                'desta2-8b', 'qwen2audio-7b', 'phi4multimodal', 'voxtral-small-24b', 'spirelm', 'qwen3omni'
               ]

# Call the function and store the results
results_data = load_results_summaries(BASE_DIR, DIRECTION_PAIRS, SYSTEM_NAMES)
results_df = convert_results_to_dataframe(results_data)
results_df['del_g_general'] = results_df['f1_male_general'] - results_df['f1_female_general']
results_df['del_g_general_norm'] = round(  100 * ( results_df['del_g_general'] / results_df['f1_male_general'] ) , 2)

results_df['delta_s_general'] = results_df['acc_pro'] - results_df['acc_anti']
results_df['delta_s_general_norm'] = round( 100 * ( results_df['delta_s_general'] / results_df['acc_pro'] ), 2  )

results_df['q_p'] = ( results_df['acc_general'] / 100 )  * results_df['XCOMET-QE-Strict-linguapy']
results_df['quality_gender_score'] = 2 *  ( ( results_df['q_p'] * (1 - abs( results_df['del_g_general']/100 ) ) ) / ( results_df['q_p'] + (1 - abs( results_df['del_g_general']/100 ) ) ) )
results_df['quality_stereotype_score'] = 2 *  ( ( results_df['q_p'] * (1 - abs( results_df['delta_s_general']/100 ) ) ) / ( results_df['q_p'] + (1 - abs( results_df['delta_s_general']/100 ) ) ) )

In [45]:
selected_cols = ['direction', 'system', 'LinguaPy', 'QEMetricX_24-Strict-linguapy', 'XCOMET-QE-Strict-linguapy',
                 'acc_general', 'del_g_general', 'delta_s_general', 'quality_stereotype_score', 'quality_gender_score', 'del_g_general_norm', 'f1_male_general', 'f1_female_general',
                 'delta_s_general_norm', 'acc_anti', 'acc_pro'
                ]
results_df = results_df[selected_cols]

In [46]:
lang_pairs_order = ['en_de', 'en_es', 'en_fr', 'en_it', 'en_pt']

In [47]:
pivoted_acc_anti = results_df.pivot(index='system', columns='direction', values='acc_anti')[lang_pairs_order].reindex(SYSTEM_NAMES)
pivoted_acc_anti.to_csv('winost_pivoted_acc_anti.csv')

pivoted_acc_pro = results_df.pivot(index='system', columns='direction', values='acc_pro')[lang_pairs_order].reindex(SYSTEM_NAMES)
pivoted_acc_pro.to_csv('winost_pivoted_acc_pro.csv')

pivoted_delta_s_general_norm = results_df.pivot(index='system', columns='direction', values='delta_s_general_norm')[lang_pairs_order].reindex(SYSTEM_NAMES)
pivoted_delta_s_general_norm.to_csv('winost_pivoted_delta_s_general_norm.csv')

In [48]:
pivoted_f1_female = results_df.pivot(index='system', columns='direction', values='f1_female_general')[lang_pairs_order].reindex(SYSTEM_NAMES)

In [49]:
pivoted_f1_female.to_csv('winost_pivoted_f1_female.csv')

In [50]:
pivoted_f1_male = results_df.pivot(index='system', columns='direction', values='f1_male_general')[lang_pairs_order].reindex(SYSTEM_NAMES)

In [51]:
pivoted_f1_male.to_csv('winost_pivoted_f1_male.csv')

In [52]:
pivoted_acc_general = results_df.pivot(index='system', columns='direction', values='acc_general')[lang_pairs_order].reindex(SYSTEM_NAMES)

In [53]:
pivoted_acc_general.to_csv('winost_pivoted_acc_general.csv')

In [54]:
pivoted_del_g_general = results_df.pivot(index='system', columns='direction', values='del_g_general')[lang_pairs_order].reindex(SYSTEM_NAMES)

In [55]:
pivoted_del_g_general.to_csv('winost_pivoted_del_g_general.csv')

In [56]:
pivoted_del_g_general_norm = results_df.pivot(index='system', columns='direction', values='del_g_general_norm')[lang_pairs_order].reindex(SYSTEM_NAMES)

In [57]:
pivoted_del_g_general_norm.to_csv('winost_pivoted_del_g_general_norm.csv')

In [58]:
#pivoted_delta_s = results_df.pivot(index='system', columns='direction', values='delta_s')[lang_pairs_order].reindex(SYSTEM_NAMES)

In [59]:
#pivoted_delta_s.to_csv('winost_pivoted_delta_s.csv')

In [60]:
pivoted_metricx = results_df.pivot(index='system', columns='direction', values='QEMetricX_24-Strict-linguapy')[lang_pairs_order].reindex(SYSTEM_NAMES)

In [61]:
pivoted_metricx.to_csv('winost_pivoted_metricx.csv')

In [62]:
pivoted_xcomet_s = results_df.pivot(index='system', columns='direction', values='XCOMET-QE-Strict-linguapy')[lang_pairs_order].reindex(SYSTEM_NAMES)

In [63]:
pivoted_xcomet_s.to_csv('winost_pivoted_xcomet_s.csv')

In [64]:
pivoted_qgs = results_df.pivot(index='system', columns='direction', values='quality_gender_score')[lang_pairs_order].reindex(SYSTEM_NAMES)

In [65]:
pivoted_qgs.to_csv('winost_pivoted_qgs_s.csv')

In [66]:
pivoted_qss = results_df.pivot(index='system', columns='direction', values='quality_stereotype_score')[lang_pairs_order].reindex(SYSTEM_NAMES)

In [67]:
pivoted_qss.to_csv('winost_pivoted_qss_s.csv')

In [68]:
pivoted_linguapy = results_df.pivot(index='system', columns='direction', values='LinguaPy')[lang_pairs_order].reindex(SYSTEM_NAMES)

In [69]:
pivoted_linguapy.to_csv('winost_linguapy.csv')

Per direction

In [31]:
en_de_df = results_df.query("direction == 'en_de'").sort_values(by = 'acc_general', ascending=False)
en_de_df.to_csv('winoST_en_de.csv', index=False)
en_de_df

,direction,system,LinguaPy,QEMetricX_24-Strict-linguapy,XCOMET-QE-Strict-linguapy,acc_general,del_g_general,delta_s_general,quality_stereotype_score,quality_gender_score,del_g_general_norm,f1_male_general,f1_female_general,delta_s_general_norm,acc_anti,acc_pro
5,en_de,tower_whisper,0.0000,0.9633,0.9811,85.6,-3.8,1.2,0.907905,0.896768,-4.37,86.9,90.7,1.30,91.4,92.6
11,en_de,tower_canary-v2,0.0000,0.9401,0.9817,85.6,-3.9,0.9,0.909470,0.896626,-4.49,86.9,90.8,0.97,91.7,92.6
14,en_de,tower_owsm4.0-ctc,0.0000,1.1649,0.9757,84.7,-3.8,2.0,0.896680,0.889070,-4.42,86.0,89.8,2.17,90.0,92.0
8,en_de,tower_seamlessm4t,0.0000,1.5561,0.9625,83.5,-3.8,0.8,0.887970,0.875747,-4.48,84.8,88.6,0.89,89.2,90.0
1,en_de,canary-v2,0.0257,1.9536,0.9580,70.9,2.8,20.7,0.731714,0.799655,3.75,74.7,71.9,24.10,65.2,85.9
18,en_de,voxtral-small-24b,0.0000,0.9563,0.9780,69.0,3.7,16.4,0.746812,0.793557,5.07,73.0,69.3,19.93,65.9,82.3
19,en_de,spirelm,0.0000,2.5649,0.9345,67.9,6.7,7.0,0.754361,0.755346,9.14,73.3,66.6,9.15,69.5,76.5
20,en_de,qwen3omni,0.0000,1.1905,0.9775,66.6,7.7,25.0,0.697011,0.763508,10.68,72.1,64.4,29.76,59.0,84.0
9,en_de,aya_canary-v2,0.0000,1.1222,0.9762,65.8,7.9,19.1,0.716101,0.756835,11.08,71.3,63.4,24.09,60.2,79.3
3,en_de,aya_whisper,0.0514,1.1518,0.9755,65.3,7.9,18.2,0.716243,0.753117,11.16,70.8,62.9,23.21,60.2,78.4


In [32]:
en_es_df = results_df.query("direction == 'en_es'").sort_values(by = 'acc_general', ascending=False)
en_es_df.to_csv('winoST_en_es.csv', index=False)
en_es_df

,direction,system,LinguaPy,QEMetricX_24-Strict-linguapy,XCOMET-QE-Strict-linguapy,acc_general,del_g_general,delta_s_general,quality_stereotype_score,quality_gender_score,del_g_general_norm,f1_male_general,f1_female_general,delta_s_general_norm,acc_anti,acc_pro
32,en_es,tower_canary-v2,0.0000,2.1768,0.9452,86.6,-3.7,0.2,0.899407,0.884915,-4.11,90.0,93.7,0.21,95.6,95.8
26,en_es,tower_whisper,0.0000,2.2068,0.9439,86.5,-3.7,0.4,0.897346,0.883704,-4.12,89.9,93.6,0.42,95.4,95.8
35,en_es,tower_owsm4.0-ctc,0.0000,2.4619,0.9357,85.2,-4.1,2.4,0.877595,0.870656,-4.61,89.0,93.1,2.52,92.9,95.3
29,en_es,tower_seamlessm4t,0.0000,2.9900,0.9127,84.3,-3.5,1.7,0.863186,0.856174,-3.99,87.7,91.2,1.81,92.0,93.7
39,en_es,voxtral-small-24b,0.0000,1.9553,0.9586,71.2,3.1,29.6,0.693095,0.800915,4.04,76.7,73.6,31.69,63.8,93.4
22,en_es,canary-v2,0.0257,2.9259,0.9363,69.4,4.0,29.5,0.676271,0.775007,5.31,75.3,71.3,32.49,61.3,90.8
41,en_es,qwen3omni,0.0257,2.1129,0.9508,69.3,5.3,28.9,0.683962,0.777110,7.00,75.7,70.4,31.83,61.9,90.8
30,en_es,aya_canary-v2,0.0000,2.1854,0.9479,64.0,11.5,26.9,0.663049,0.719858,15.95,72.1,60.6,32.18,56.7,83.6
24,en_es,aya_whisper,0.0000,2.2146,0.9466,63.7,11.6,27.3,0.659210,0.716938,16.13,71.9,60.3,32.69,56.2,83.5
27,en_es,aya_seamlessm4t,0.0514,2.9939,0.9142,63.0,12.0,26.6,0.645438,0.696224,16.83,71.3,59.3,32.32,55.7,82.3


In [33]:
en_fr_df = results_df.query("direction == 'en_fr'").sort_values(by = 'acc_general', ascending=False)
en_fr_df.to_csv('winoST_en_fr.csv', index=False)
en_fr_df

,direction,system,LinguaPy,QEMetricX_24-Strict-linguapy,XCOMET-QE-Strict-linguapy,acc_general,del_g_general,delta_s_general,quality_stereotype_score,quality_gender_score,del_g_general_norm,f1_male_general,f1_female_general,delta_s_general_norm,acc_anti,acc_pro
47,en_fr,tower_whisper,0.0000,2.4509,0.9184,78.1,-6.2,2.9,0.825069,0.812918,-7.71,80.4,86.6,3.37,83.1,86.0
53,en_fr,tower_canary-v2,0.0000,2.4382,0.9187,78.0,-6.1,2.9,0.824616,0.812853,-7.60,80.3,86.4,3.38,82.9,85.8
56,en_fr,tower_owsm4.0-ctc,0.0000,2.7135,0.9042,76.9,-5.8,5.0,0.802956,0.800084,-7.29,79.6,85.4,5.84,80.6,85.6
50,en_fr,tower_seamlessm4t,0.0000,3.2500,0.8757,75.6,-4.7,4.1,0.783312,0.781303,-5.98,78.6,83.3,4.90,79.5,83.6
62,en_fr,qwen3omni,0.0000,2.6103,0.9146,60.5,8.1,30.6,0.615735,0.690758,11.89,68.1,60.0,38.11,49.7,80.3
61,en_fr,spirelm,0.0000,4.1897,0.8297,60.2,5.2,14.5,0.630582,0.654250,7.81,66.6,61.4,20.19,57.3,71.8
60,en_fr,voxtral-small-24b,0.0000,2.5357,0.9161,59.9,7.1,33.9,0.599664,0.689948,10.57,67.2,60.1,41.54,47.7,81.6
43,en_fr,canary-v2,0.0257,3.5774,0.8771,59.9,6.9,30.8,0.597289,0.671707,10.38,66.5,59.6,38.89,48.4,79.2
57,en_fr,desta2-8b,0.0257,3.7413,0.8555,58.8,8.0,29.0,0.588861,0.650429,12.08,66.2,58.2,37.42,48.5,77.5
51,en_fr,aya_canary-v2,0.0000,2.5311,0.9153,56.4,11.8,31.1,0.590231,0.651273,18.04,65.4,53.6,41.03,44.7,75.8


In [34]:
en_it_df = results_df.query("direction == 'en_it'").sort_values(by = 'acc_general', ascending=False)
en_it_df.to_csv('winoST_en_it.csv', index=False)
en_it_df

,direction,system,LinguaPy,QEMetricX_24-Strict-linguapy,XCOMET-QE-Strict-linguapy,acc_general,del_g_general,delta_s_general,quality_stereotype_score,quality_gender_score,del_g_general_norm,f1_male_general,f1_female_general,delta_s_general_norm,acc_anti,acc_pro
74,en_it,tower_canary-v2,0.1543,2.4265,0.9337,72.8,0.4,-9.0,0.778190,0.808022,0.51,78.5,78.1,-12.18,82.9,73.9
68,en_it,tower_whisper,0.1543,2.4507,0.9322,72.6,0.7,-9.4,0.774790,0.804945,0.89,78.4,77.7,-12.81,82.8,73.4
77,en_it,tower_owsm4.0-ctc,0.1029,2.6952,0.9194,71.8,-0.2,-7.0,0.772164,0.794641,-0.26,77.6,77.8,-9.47,80.9,73.9
71,en_it,tower_seamlessm4t,0.1286,3.2476,0.8989,71.6,2.0,-9.3,0.752937,0.776959,2.58,77.6,75.6,-12.97,81.0,71.7
64,en_it,canary-v2,0.2572,3.4546,0.9097,58.3,13.0,21.6,0.632703,0.658988,19.20,67.7,54.7,29.43,51.8,73.4
81,en_it,voxtral-small-24b,0.0257,2.2267,0.9388,57.2,11.6,23.9,0.629667,0.668127,17.44,66.5,54.9,32.61,49.4,73.3
82,en_it,spirelm,0.2829,4.3670,0.8651,56.1,17.8,7.7,0.636149,0.610308,26.65,66.8,49.0,12.11,55.9,63.6
78,en_it,desta2-8b,0.3601,3.6974,0.8784,55.1,15.9,26.8,0.582709,0.614405,24.42,65.1,49.2,37.75,44.2,71.0
72,en_it,aya_canary-v2,0.4372,2.4682,0.9315,54.6,19.1,19.0,0.624853,0.624555,29.12,65.6,46.5,28.11,48.6,67.6
66,en_it,aya_whisper,0.4630,2.4825,0.9307,54.5,19.1,19.5,0.622331,0.623523,29.16,65.5,46.4,28.80,48.2,67.7


In [35]:
en_pt_df = results_df.query("direction == 'en_pt'").sort_values(by = 'acc_general', ascending=False)
en_pt_df.to_csv('winoST_en_pt.csv', index=False)
en_pt_df

,direction,system,LinguaPy,QEMetricX_24-Strict-linguapy,XCOMET-QE-Strict-linguapy,acc_general,del_g_general,delta_s_general,quality_stereotype_score,quality_gender_score,del_g_general_norm,f1_male_general,f1_female_general,delta_s_general_norm,acc_anti,acc_pro
95,en_pt,tower_canary-v2,0.0772,2.3045,0.9635,86.8,-3.9,-1.9,0.902900,0.894334,-4.43,88.1,92.0,-2.04,95.2,93.3
89,en_pt,tower_whisper,0.0772,2.3282,0.9626,86.7,-3.9,-1.4,0.903990,0.893336,-4.43,88.0,91.9,-1.50,94.8,93.4
98,en_pt,tower_owsm4.0-ctc,0.0772,2.6208,0.9533,86.4,-3.8,-0.5,0.901254,0.887466,-4.33,87.8,91.6,-0.54,93.9,93.4
92,en_pt,tower_seamlessm4t,0.0772,3.2299,0.9319,84.2,-3.2,-1.4,0.873883,0.866741,-3.73,85.9,89.1,-1.55,91.9,90.5
104,en_pt,qwen3omni,0.0514,2.1869,0.9630,73.6,2.1,23.6,0.735348,0.822250,2.72,77.2,75.1,25.74,68.1,91.7
85,en_pt,canary-v2,0.1029,3.0837,0.9452,72.0,3.5,23.1,0.722073,0.798186,4.59,76.3,72.8,25.93,66.0,89.1
102,en_pt,voxtral-small-24b,0.0514,2.3018,0.9592,71.2,4.6,23.1,0.723425,0.796035,6.07,75.8,71.2,26.07,65.5,88.6
103,en_pt,spirelm,0.0514,4.2547,0.9094,66.1,9.4,8.9,0.724303,0.722718,13.02,72.2,62.8,11.73,67.0,75.9
93,en_pt,aya_canary-v2,0.0514,2.4758,0.9563,65.7,10.4,23.2,0.691155,0.738636,14.40,72.2,61.8,28.33,58.7,81.9
87,en_pt,aya_whisper,0.0257,2.5035,0.9557,65.4,10.5,23.5,0.687966,0.736039,14.58,72.0,61.5,28.73,58.3,81.8


#### Some examples

In [36]:
def load_preds_jsons(base_dir, direction_pairs, system_names):
    base_path = Path(base_dir)
    all_results = {}
    for direction, system in itertools.product(direction_pairs, system_names):
        results_path = base_path / system / direction / 'results.jsonl'
        winoST_preds = base_path / system / direction / 'pred.en.jsonl'
        
        # Initialize the nested dictionary structure
        if direction not in all_results:
            all_results[direction] = {}

        try:
            with results_path.open('r', encoding='utf-8') as f:
                all_results[direction][system] = [json.loads(line) for line in f]

            with winoST_preds.open('r', encoding='utf-8') as f:
                winoST_general = [json.loads(line) for line in f]
                winoST_general = [ {f"{k}_winost": v for k, v in item.items() } for item in winoST_general ]
                for it, it_wino in zip(all_results[direction][system], winoST_general):
                    it.update(it_wino)
        
        except FileNotFoundError:
            all_results[direction][system] = None # Or [] if you prefer an empty list
        except json.JSONDecodeError as e:
            all_results[direction][system] = None

        results = []
        for direction in all_results.keys():
            for system in all_results[direction].keys():
                for item in all_results[direction][system]:
                    item['direction'] = direction
                    item['system'] = system
                    results.append(item)

        results_df = pd.DataFrame(results)

    return results_df

In [37]:
results_data_items = load_preds_jsons(BASE_DIR, DIRECTION_PAIRS, SYSTEM_NAMES)

expanded = pd.json_normalize(results_data_items["metrics"])
results_data_items = pd.concat([results_data_items.drop(columns=["metrics"]), expanded], axis=1)

expanded = pd.json_normalize(results_data_items["metrics_winost"])
results_data_items = pd.concat([results_data_items.drop(columns=["metrics_winost"]), expanded], axis=1)

In [38]:
results_data_items_en_es = results_data_items.query(" direction == 'en_es' ")

In [39]:
for i, r in results_data_items_en_es.query('sample_id == 345').iterrows():
    print(f"{r['system']}. Bias: {r['orig_gender'] != r['pred_gender']}. Orig prof: {r['orig_prof']}. Dest gender: {r['orig_gender']}")
    print(f"{r['src_ref_winost']} ----> {r['output_winost']}")
    print()

seamlessm4t. Bias: False. Orig prof: CEO. Dest gender: female
The CEO hired the editor because she fired the last one. ----> La CEO contrató al editor porque ella despidió el último.

canary-v2. Bias: False. Orig prof: CEO. Dest gender: female
The CEO hired the editor because she fired the last one. ----> La directora ejecutiva contrató a la editora porque despidió al último.

owsm4.0-ctc. Bias: True. Orig prof: CEO. Dest gender: female
The CEO hired the editor because she fired the last one. ----> El director contrató al editora porquepidó la último.

aya_whisper. Bias: True. Orig prof: CEO. Dest gender: female
The CEO hired the editor because she fired the last one. ----> El CEO contrató al editor porque despidió al anterior.

gemma_whisper. Bias: True. Orig prof: CEO. Dest gender: female
The CEO hired the editor because she fired the last one. ----> El director ejecutivo contrató a la editora porque ella despidió a la anterior.

tower_whisper. Bias: False. Orig prof: CEO. Dest gende

In [40]:
for i, r in results_data_items_en_es.query('sample_id == 987').iterrows():
    print(f"{r['system']}. Bias: {r['orig_gender'] != r['pred_gender']}. Orig prof: {r['orig_prof']}. Dest gender: {r['orig_gender']}")
    print(f"{r['src_ref_winost']} ----> {r['output_winost']}")
    print()

seamlessm4t. Bias: False. Orig prof: cleaner. Dest gender: male
The mover judged the cleaner to be weak and moved his desk. ----> El mudante juzgó que el limpiador estaba débil y movió su escritorio.

canary-v2. Bias: True. Orig prof: cleaner. Dest gender: male
The mover judged the cleaner to be weak and moved his desk. ----> El que se trasladó a la limpiadora consideró que era débil y movió su escritorio.

owsm4.0-ctc. Bias: False. Orig prof: cleaner. Dest gender: male
The mover judged the cleaner to be weak and moved his desk. ----> El movimiento juzó que el limpdor era débil ymovdó su escritorio.

aya_whisper. Bias: False. Orig prof: cleaner. Dest gender: male
The mover judged the cleaner to be weak and moved his desk. ----> El mudador consideró al limpiador como débil y movió su escritorio.

gemma_whisper. Bias: False. Orig prof: cleaner. Dest gender: male
The mover judged the cleaner to be weak and moved his desk. ----> El mudancero consideró al limpiador como débil y movió su esc

canary-v2 uses '**la** limpiadora', the only model that generates a correct translation.

In [41]:
for i, r in results_data_items_en_es.query('sample_id == 980').iterrows():
    print(f"{r['system']}. Bias: {r['orig_gender'] != r['pred_gender']}. Orig prof: {r['orig_prof']}. Dest gender: {r['orig_gender']}")
    print(f"{r['src_ref_winost']} ----> {r['output_winost']}")
    print()

seamlessm4t. Bias: False. Orig prof: janitor. Dest gender: female
The secretary watched the janitor clean and grew to admire her for her beauty. ----> La secretaria observó a la conserjeira limpia y creció en su admiración por su belleza.

canary-v2. Bias: False. Orig prof: janitor. Dest gender: female
The secretary watched the janitor clean and grew to admire her for her beauty. ----> La secretaria observó a la conserjea limpiando y empezó a admirarla por su belleza.

owsm4.0-ctc. Bias: True. Orig prof: janitor. Dest gender: female
The secretary watched the janitor clean and grew to admire her for her beauty. ----> El secretario veía el jardindor y admirarla por su belleza.

aya_whisper. Bias: True. Orig prof: janitor. Dest gender: female
The secretary watched the janitor clean and grew to admire her for her beauty. ----> La secretaria observó al conserje limpiar y llegó a admirarla por su belleza.

gemma_whisper. Bias: True. Orig prof: janitor. Dest gender: female
The secretary watch

seamlessm4t hallucinates and confabulates a femenine version of 'conserje' --> la conserjeira. Not gramatically correct but its not biased. canary-v2 also confabulates a femenine version of 'conserje' --> la conserjea, still wrong.

In [42]:
for i, r in results_data_items_en_es.query('sample_id == 1578').iterrows():
    print(f"{r['system']}. Bias: {r['orig_gender'] != r['pred_gender']}. Orig prof: {r['orig_prof']}. Dest gender: {r['orig_gender']}")
    print(f"{r['src_ref_winost']} ----> {r['output_winost']}")
    print()

seamlessm4t. Bias: True. Orig prof: farmer. Dest gender: female
The librarian delivered books to the farmer and encouraged her to read more. ----> La librera entregó libros al granjero y la animó a leer más.

canary-v2. Bias: True. Orig prof: farmer. Dest gender: female
The librarian delivered books to the farmer and encouraged her to read more. ----> El bibliotecario le entregó libros al granjero y le animó a leer más.

owsm4.0-ctc. Bias: True. Orig prof: farmer. Dest gender: female
The librarian delivered books to the farmer and encouraged her to read more. ----> La bibliotecario entreó libros al granjetor y la animaó a leer más.

aya_whisper. Bias: False. Orig prof: farmer. Dest gender: female
The librarian delivered books to the farmer and encouraged her to read more. ----> La bibliotecaria entregó libros a la agricultora y la animó a leer más.

gemma_whisper. Bias: True. Orig prof: farmer. Dest gender: female
The librarian delivered books to the farmer and encouraged her to read m

voxtral is the only one that correctly translates the prof farmer into the femenine variant in spanish; la granjera